# Individual Exercise: Data Quality Audit

Homework notebook for Week 2.

**Name:** Vinay Bommenahalli Gurubasappa  
**Dataset:** *week2_customer_transactions_messy.csv*


## Instructions

Perform a mini data quality audit and submit this completed notebook.


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

# Load dataset — try repo-relative path first, fall back to current directory
_candidates = [
    Path.cwd().resolve().parents[1] / 'data' / 'raw' / 'week2_customer_transactions_messy.csv',
    Path.cwd() / 'week2_customer_transactions_messy.csv',
    Path('week2_customer_transactions_messy.csv'),
]
data_path = next((p for p in _candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("CSV not found. Place it alongside this notebook or in data/raw/.")

df = pd.read_csv(data_path)
print("Loaded:", data_path)
print("Shape:", df.shape)
df.head(11)


Loaded: c:\Users\Vinay\OneDrive - SRH\SRH\Data Mangement\Individual Assignment\Week 2\Exercise_1_Individual_Data_Quality_Audit\week2_customer_transactions_messy.csv
Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
7,T0007,C106,2026-02-30,47.00,EUR,card,completed,DE,2026-02-15
8,T0008,C107,2026-01-10,1000000.00,EUR,card,completed,DE,2026-01-10
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN


## Required tasks

1. Describe the dataset briefly  
2. Identify issues by dimension  
3. Compute at least 3 KPIs  
4. Define at least 3 validation rules  
5. Create a short audit summary  
6. Recommend cleaning steps for next chapter (do not implement full pipelines)


## Task 1 – Dataset Description

### Overview

The dataset `week2_customer_transactions_messy.csv` contains **11 rows** and **9 columns** representing
individual financial transactions recorded by a (fictional) e-commerce or payment platform.

| Column | Description |
|---|---|
| `transaction_id` | Unique identifier for each transaction (e.g., T0001) |
| `customer_id` | Identifier of the customer who made the transaction |
| `transaction_date` | Date the transaction occurred |
| `amount` | Monetary value of the transaction |
| `currency` | ISO 4217 currency code (expected: EUR, USD, GBP …) |
| `payment_method` | Method used (card, bank_transfer, cash) |
| `status` | Transaction state: completed / pending / cancelled |
| `region` | ISO country code of the customer (DE, FR, US, NL …) |
| `last_updated` | Date the record was last modified in the system |

### Possible Business Use Case

This dataset could serve a **payment operations team** or **finance analytics function** to:
- Track daily transaction volumes and revenue per region
- Monitor payment method distribution and failure rates
- Feed downstream dashboards for KPI reporting (e.g., monthly revenue by country)
- Trigger fraud or anomaly alerts for unusually large or negative transactions

Data quality is critical here because errors in `amount`, `currency`, or `status` directly
distort revenue figures and can lead to incorrect financial reporting.


## Task 2 – Issues by Dimension

The table below maps every identified data quality issue to a standard DQ dimension.


In [8]:
issue_table = pd.DataFrame([
    # Issue                                  Dimension       Affected Column(s)          Evidence / Row(s)                                Impact
    ["Missing customer_id",                  "Completeness", "customer_id",              "Row T0004",                                     "Cannot link transaction to a customer"],
    ["Missing amount",                       "Completeness", "amount",                   "Row T0009",                                     "Revenue calculation fails"],
    ["Missing payment_method",               "Completeness", "payment_method",           "Row T0010",                                     "Payment-method analysis incomplete"],
    ["Missing last_updated",                 "Completeness", "last_updated",             "Row T0009",                                     "Audit trail broken"],
    ["Duplicate transaction_id",             "Uniqueness",   "transaction_id",           "T0006 appears twice (rows 5–6)",                "Double-counts revenue; violates PK constraint"],
    ["Negative amount",                      "Validity",     "amount",                   "Row T0003: -35.00",                             "Invalid unless refund; status says completed"],
    ["Invalid date (Feb 30)",                "Validity",     "transaction_date",         "Row T0007: 2026-02-30",                         "Non-existent date; parse returns NaT"],
    ["Non-standard currency code",           "Validity",     "currency",                 "Row T0005: EURO instead of EUR",                "ISO-4217 violation; join with fx-rate table fails"],
    ["Inconsistent date format",             "Consistency",  "transaction_date",         "YYYY-MM-DD vs YYYY/MM/DD vs DD-MM-YYYY",        "Parsing errors; wrong date ordering"],
    ["Inconsistent case – region",           "Consistency",  "region",                   "Row T0002: 'de' should be 'DE'",               "GROUP BY region returns split counts"],
    ["Inconsistent case – payment_method",   "Consistency",  "payment_method",           "Row T0002: 'CARD' should be 'card'",           "Categorical mismatch in aggregations"],
    ["Suspiciously large amount (outlier)",  "Validity",     "amount",                   "Row T0008: 1,000,000 EUR",                     "Likely data-entry error; distorts mean revenue"],
    ["last_updated before transaction_date", "Integrity",    "last_updated,transaction_date","Row T0003 (ambiguous due to date format)",  "Temporal integrity violation; suggests data pipeline issue"],
], columns=["Issue", "Dimension", "Affected Column(s)", "Evidence / Row(s)", "Impact"])

issue_table


,Issue,Dimension,Affected Column(s),Evidence / Row(s),Impact
0,Missing customer_id,Completeness,customer_id,Row T0004,Cannot link transaction to a customer
1,Missing amount,Completeness,amount,Row T0009,Revenue calculation fails
2,Missing payment_method,Completeness,payment_method,Row T0010,Payment-method analysis incomplete
3,Missing last_updated,Completeness,last_updated,Row T0009,Audit trail broken
4,Duplicate transaction_id,Uniqueness,transaction_id,T0006 appears twice (rows 5–6),Double-counts revenue; violates PK constraint
5,Negative amount,Validity,amount,Row T0003: -35.00,Invalid unless refund; status says completed
6,Invalid date (Feb 30),Validity,transaction_date,Row T0007: 2026-02-30,Non-existent date; parse returns NaT
7,Non-standard currency code,Validity,currency,Row T0005: EURO instead of EUR,ISO-4217 violation; join with fx-rate table fails
8,Inconsistent date format,Consistency,transaction_date,YYYY-MM-DD vs YYYY/MM/DD vs DD-MM-YYYY,Parsing errors; wrong date ordering
9,Inconsistent case – region,Consistency,region,Row T0002: 'de' should be 'DE',GROUP BY region returns split counts


## Task 3 – KPI Calculations

Five KPIs are computed below to quantify the dataset's quality profile.


In [9]:
# ── helpers ──────────────────────────────────────────────────────────────────
amount_num   = pd.to_numeric(df['amount'], errors='coerce')
date_parsed  = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
VALID_CCY    = {'EUR', 'USD', 'GBP', 'CHF', 'GBP'}

total_cells  = df.shape[0] * df.shape[1]

# ── KPI 1: Completeness Rate ─────────────────────────────────────────────────
completeness_rate = 1 - (df.isna().sum().sum() / total_cells)

# ── KPI 2: Duplication Rate ──────────────────────────────────────────────────
duplication_rate = df.duplicated(subset=['transaction_id']).mean()

# ── KPI 3: Error Rate ────────────────────────────────────────────────────────
# A row is erroneous if: amount is NaN OR amount < 0 OR date cannot be parsed
error_mask   = amount_num.isna() | (amount_num < 0) | date_parsed.isna()
error_rate   = error_mask.mean()

# ── KPI 4: Currency Validity Rate ────────────────────────────────────────────
currency_validity_rate = df['currency'].isin(VALID_CCY).mean()

# ── KPI 5: Outlier Rate (IQR method on amount) ───────────────────────────────
q1, q3  = amount_num.quantile(0.25), amount_num.quantile(0.75)
iqr     = q3 - q1
outlier_mask         = (amount_num < (q1 - 1.5 * iqr)) | (amount_num > (q3 + 1.5 * iqr))
outlier_rate         = outlier_mask.mean()

kpi_df = pd.DataFrame({
    'KPI':   ['Completeness Rate', 'Duplication Rate', 'Error Rate',
              'Currency Validity Rate', 'Outlier Rate (IQR, amount)'],
    'Value': [completeness_rate, duplication_rate, error_rate,
              currency_validity_rate, outlier_rate],
    'Value %': [f"{v*100:.1f}%" for v in [completeness_rate, duplication_rate, error_rate,
                                           currency_validity_rate, outlier_rate]],
})
kpi_df


,KPI,Value,Value %
0,Completeness Rate,0.959596,96.0%
1,Duplication Rate,0.090909,9.1%
2,Error Rate,0.272727,27.3%
3,Currency Validity Rate,0.909091,90.9%
4,"Outlier Rate (IQR, amount)",0.090909,9.1%


### KPI Interpretation

| KPI | Value | Interpretation |
|---|---|---|
| **Completeness Rate** | 95.96% | 4 of 99 cells are null — missing `customer_id`, `amount`, `payment_method`, `last_updated`. Relatively high completeness but critical fields are affected. |
| **Duplication Rate** | 9.09% | 1 in 11 transaction IDs is a duplicate (T0006). In financial data any duplication is unacceptable as it double-counts revenue. |
| **Error Rate** | 27.27% | 3 of 11 rows contain a format or value error (negative amount, invalid date, or unparseable date). This is critically high for a payment dataset. |
| **Currency Validity Rate** | 90.91% | 1 row uses `EURO` instead of ISO-4217 `EUR`. Minor but causes join failures with FX-rate reference tables. |
| **Outlier Rate (IQR)** | 9.09% | Row T0008 with amount = 1,000,000 EUR is flagged as a statistical outlier. Needs business-rule verification (possible fat-finger entry). |

**Strongest signal:** The **Error Rate at 27.27%** is the most alarming KPI — over a quarter of records
carry structural errors that would break downstream pipelines. The **Duplication Rate of 9.09%** is the
highest-priority issue to escalate because duplicate transactions directly corrupt financial totals.


## Task 4 – Validation Rules

Six rules are defined below. Each rule returns a boolean mask where `True` = violation detected.


In [10]:
amount_num  = pd.to_numeric(df['amount'], errors='coerce')
date_parsed = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
VALID_CCY   = {'EUR', 'USD', 'GBP', 'CHF'}
VALID_PM    = {'card', 'bank_transfer', 'cash', 'wallet'}
VALID_ST    = {'completed', 'pending', 'cancelled', 'failed'}

rules = {
    # Rule 1 – Completeness: customer_id must not be null
    'RULE-01_customer_id_required':
        df['customer_id'].isna() | (df['customer_id'].astype(str).str.strip() == ''),

    # Rule 2 – Validity: amount must be a non-negative number
    'RULE-02_amount_non_negative':
        amount_num.isna() | (amount_num < 0),

    # Rule 3 – Validity: transaction_date must parse to a real calendar date
    'RULE-03_transaction_date_parseable':
        date_parsed.isna(),

    # Rule 4 – Uniqueness: transaction_id must be unique across all rows
    'RULE-04_transaction_id_unique':
        df.duplicated(subset=['transaction_id'], keep=False),

    # Rule 5 – Validity: currency must be a recognised ISO-4217 code (upper-case)
    'RULE-05_currency_valid_iso4217':
        ~df['currency'].isin(VALID_CCY),

    # Rule 6 – Consistency: region must be upper-case ISO country code
    'RULE-06_region_uppercase':
        df['region'].str.upper() != df['region'],
}

rule_summary = pd.DataFrame({
    'rule_name':     list(rules.keys()),
    'affected_rows': [int(mask.sum()) for mask in rules.values()],
    'example_rows':  [
        df.index[mask].tolist() if mask.sum() > 0 else []
        for mask in rules.values()
    ],
})
rule_summary


,rule_name,affected_rows,example_rows
0,RULE-01_customer_id_required,1,[3]
1,RULE-02_amount_non_negative,2,"[2, 9]"
2,RULE-03_transaction_date_parseable,1,[7]
3,RULE-04_transaction_id_unique,2,"[5, 6]"
4,RULE-05_currency_valid_iso4217,1,[4]
5,RULE-06_region_uppercase,1,[1]


### Validation Rule Notes

- **RULE-01** catches the missing customer_id on row index 3 (T0004).
- **RULE-02** catches the negative amount on row index 2 (T0003, -35.00) and the missing amount on row index 9 (T0009).
- **RULE-03** catches the non-existent date 2026-02-30 on row index 7 (T0007).
- **RULE-04** identifies both copies of the duplicate T0006 (row indices 5 and 6).
- **RULE-05** catches `EURO` on row index 4 (T0005).
- **RULE-06** catches `de` (lowercase) on row index 1 (T0002).


## Task 5 – Audit Summary


In [11]:
audit_summary = pd.DataFrame([
    #  issue_type                      affected_rows  severity   dimension       recommended_next_action
    ["Duplicate transaction_id",               2,    "Critical", "Uniqueness",   "Deduplicate by transaction_id; keep latest last_updated; escalate to data engineering"],
    ["Invalid / non-parseable date",           1,    "Critical", "Validity",     "Remove or quarantine row T0007; investigate source system date generation"],
    ["Negative amount",                        1,    "High",     "Validity",     "Flag as potential refund; verify with business; if not refund, correct to positive or remove"],
    ["Missing customer_id",                    1,    "High",     "Completeness", "Attempt lookup from CRM by transaction context; if not found, tag as anonymous"],
    ["Missing amount",                         1,    "High",     "Completeness", "Attempt recovery from payment gateway logs; otherwise exclude from revenue KPIs"],
    ["Non-standard currency code (EURO)",      1,    "Medium",   "Validity",     "Standardise to ISO-4217 EUR using a mapping dictionary during ingestion"],
    ["Inconsistent date formats",              2,    "Medium",   "Consistency",  "Enforce ISO-8601 (YYYY-MM-DD) at source; normalise at ingestion layer"],
    ["Inconsistent case in region/pay-method", 2,    "Medium",   "Consistency",  "Apply .str.upper() for region and .str.lower() for payment_method"],
    ["Missing payment_method",                 1,    "Medium",   "Completeness", "Impute from customer profile history or tag as 'unknown'"],
    ["Missing last_updated",                   1,    "Low",      "Completeness", "Backfill with transaction_date as default; update pipeline to always write this field"],
    ["Outlier amount (1,000,000 EUR)",         1,    "High",     "Validity",     "Flag for manual review; apply IQR-based outlier detection in validation pipeline"],
], columns=["issue_type", "affected_rows", "severity", "dimension", "recommended_next_action"])

# Sort by severity
severity_order = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}
audit_summary['_sort'] = audit_summary['severity'].map(severity_order)
audit_summary = audit_summary.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)
audit_summary


,issue_type,affected_rows,severity,dimension,recommended_next_action
0,Duplicate transaction_id,2,Critical,Uniqueness,Deduplicate by transaction_id; keep latest las...
1,Invalid / non-parseable date,1,Critical,Validity,Remove or quarantine row T0007; investigate so...
2,Negative amount,1,High,Validity,Flag as potential refund; verify with business...
3,Missing customer_id,1,High,Completeness,Attempt lookup from CRM by transaction context...
4,Missing amount,1,High,Completeness,Attempt recovery from payment gateway logs; ot...
5,"Outlier amount (1,000,000 EUR)",1,High,Validity,Flag for manual review; apply IQR-based outlie...
6,Inconsistent date formats,2,Medium,Consistency,Enforce ISO-8601 (YYYY-MM-DD) at source; norma...
7,Non-standard currency code (EURO),1,Medium,Validity,Standardise to ISO-4217 EUR using a mapping di...
8,Inconsistent case in region/pay-method,2,Medium,Consistency,Apply .str.upper() for region and .str.lower()...
9,Missing payment_method,1,Medium,Completeness,Impute from customer profile history or tag as...


## Task 6 – Recommended Cleaning Steps for Next Chapter


### 1. Standardise Date Formats
Convert all `transaction_date` and `last_updated` values to ISO-8601 (`YYYY-MM-DD`) using
`pd.to_datetime(..., format='mixed')` followed by `.dt.strftime('%Y-%m-%d')`.
Rows with genuinely invalid dates (e.g., 2026-02-30) should be quarantined in a separate
error log table rather than silently dropped.

### 2. Deduplicate Records
Apply `df.drop_duplicates(subset=['transaction_id'], keep='last')` to remove exact duplicates.
If only `transaction_id` duplicates exist with differing values in other columns, escalate
to the source team before choosing which record to retain.

### 3. Fix Currency Codes and Casing
Map non-standard currency codes using a dictionary (e.g., `{'EURO': 'EUR'}`).
Uppercase all `currency` values with `.str.upper()`.
Apply `.str.lower()` to `payment_method` and `.str.upper()` to `region` for consistency.

### 4. Handle Missing Values
- `customer_id`: Attempt CRM lookup; otherwise assign a placeholder like `UNKNOWN`.
- `amount`: Attempt recovery from source system logs; otherwise exclude from revenue aggregations.
- `payment_method`: Impute from customer history or fill with `'unknown'`.
- `last_updated`: Default to `transaction_date` if no better source is available.

### 5. Flag and Review Outliers and Negative Amounts
Use an IQR-based filter to flag amounts outside the expected range.
Negative amounts should be reviewed: if they represent refunds, a new `transaction_type`
column should be introduced (`debit` / `credit`); otherwise they are errors to be corrected.

### 6. Enforce Schema Constraints at Ingestion
Introduce a validation layer (e.g., using `pandera` or `Great Expectations`) at the
pipeline ingestion step to enforce:
- Non-null constraints on `transaction_id`, `customer_id`, `amount`
- `currency` must match a whitelist of ISO-4217 codes
- `transaction_date` must be a valid real calendar date
- `transaction_id` must be unique


## Reflection Questions

**1. Which KPI gave the strongest signal?**  
The **Error Rate (27.27%)** gave the strongest signal. It reveals that more than one in four rows
contains a structural flaw — an unparseable date, a negative amount, or a missing value in a
critical numeric field. This would cause silent failures in any automated analytics pipeline.

**2. Which issue should be escalated first?**  
The **duplicate transaction_id (T0006)** should be escalated first because it violates a primary-key
constraint that is foundational to all downstream joins and revenue aggregations. A single duplicate
in 11 rows (9.09%) would translate to thousands of corrupted records in a production dataset of
millions. It requires both an immediate deduplication fix *and* a root-cause investigation in the
source system or ETL process.

**3. What additional metadata would improve this audit?**  
- **Expected value ranges** for `amount` from the business (e.g., min/max per transaction type)
to set data-driven outlier thresholds instead of statistical heuristics.  
- **Allowed value lists** for `currency`, `region`, `payment_method`, and `status` from a
reference/master data catalogue, enabling strict whitelist validation.  
- **Source system timestamps** or ingestion pipeline logs to distinguish between errors
introduced at the source vs. during ETL, which determines the responsible team for remediation.


## Bonus – Reusable Audit Helper Function

The `run_audit` function below packages the entire audit workflow into a single reusable
utility with a clear interface and docstring.


In [12]:
def run_audit(
    dataframe: pd.DataFrame,
    valid_currencies: set = None,
    amount_col: str = 'amount',
    date_col: str = 'transaction_date',
    id_col: str = 'transaction_id',
    iqr_multiplier: float = 1.5,
) -> dict:
    """Perform a full data quality audit on a transaction DataFrame.

    Parameters
    ----------
    dataframe : pd.DataFrame
        The input DataFrame to audit. Must contain at minimum the columns referenced
        by ``amount_col``, ``date_col``, and ``id_col``.
    valid_currencies : set, optional
        Set of accepted ISO-4217 currency codes (upper-case).
        Defaults to {'EUR', 'USD', 'GBP', 'CHF'} if not provided.
        Changing this set affects how many rows RULE-05 flags; wider sets reduce the
        Currency Validity Rate and may hide data errors.
    amount_col : str, default 'amount'
        Name of the column holding transaction amounts. Used for negativity checks
        and outlier detection. Changing to a wrong column name raises a KeyError.
    date_col : str, default 'transaction_date'
        Name of the column holding transaction dates. Used for format validation.
    id_col : str, default 'transaction_id'
        Name of the primary-key column. Used for uniqueness checks.
    iqr_multiplier : float, default 1.5
        The multiplier applied to the IQR for outlier fencing.
        Lower values (e.g., 1.0) flag more rows as outliers; higher values (e.g., 3.0)
        flag fewer. The standard Tukey fence uses 1.5.

    Returns
    -------
    dict
        Dictionary with keys 'kpis' (pd.DataFrame) and 'audit_summary' (pd.DataFrame).
    """
    if valid_currencies is None:
        valid_currencies = {'EUR', 'USD', 'GBP', 'CHF'}

    amt      = pd.to_numeric(dataframe[amount_col], errors='coerce')
    dates    = pd.to_datetime(dataframe[date_col],  errors='coerce', format='mixed')
    q1_, q3_ = amt.quantile(0.25), amt.quantile(0.75)
    iqr_     = q3_ - q1_

    kpi_vals = {
        'Completeness Rate':        1 - (dataframe.isna().sum().sum() /
                                         (dataframe.shape[0] * dataframe.shape[1])),
        'Duplication Rate':         dataframe.duplicated(subset=[id_col]).mean(),
        'Error Rate':               (amt.isna() | (amt < 0) | dates.isna()).mean(),
        'Currency Validity Rate':   dataframe['currency'].isin(valid_currencies).mean(),
        'Outlier Rate (IQR)':       ((amt < q1_ - iqr_multiplier * iqr_) |
                                     (amt > q3_ + iqr_multiplier * iqr_)).mean(),
    }

    rules_ = {
        'customer_id_required':  dataframe['customer_id'].isna(),
        'amount_non_negative':   amt.isna() | (amt < 0),
        'date_parseable':        dates.isna(),
        'transaction_id_unique': dataframe.duplicated(subset=[id_col], keep=False),
        'currency_valid':        ~dataframe['currency'].isin(valid_currencies),
        'region_uppercase':      dataframe['region'].str.upper() != dataframe['region'],
    }

    kpi_df = pd.DataFrame(kpi_vals.items(), columns=['KPI', 'Value'])
    kpi_df['Value %'] = kpi_df['Value'].map(lambda v: f"{v*100:.1f}%")

    summary_df = pd.DataFrame({
        'rule_name':     list(rules_.keys()),
        'affected_rows': [int(m.sum()) for m in rules_.values()],
    }).sort_values('affected_rows', ascending=False).reset_index(drop=True)

    return {'kpis': kpi_df, 'audit_summary': summary_df}


# ── Run on our dataset ────────────────────────────────────────────────────────
result = run_audit(df)
print("=== KPIs ===")
display(result['kpis'])
print("\n=== Validation Rule Summary ===")
display(result['audit_summary'])


=== KPIs ===


,KPI,Value,Value %
0,Completeness Rate,0.959596,96.0%
1,Duplication Rate,0.090909,9.1%
2,Error Rate,0.272727,27.3%
3,Currency Validity Rate,0.909091,90.9%
4,Outlier Rate (IQR),0.090909,9.1%



=== Validation Rule Summary ===


,rule_name,affected_rows
0,amount_non_negative,2
1,transaction_id_unique,2
2,customer_id_required,1
3,date_parseable,1
4,currency_valid,1
5,region_uppercase,1


### Function Parameter Explanation

- **`dataframe`** – The only required input; all computations operate on this DataFrame.
  Keeping it as a parameter (rather than a global) makes the function fully reusable
  across different datasets or time-period slices without modifying any internal logic.

- **`valid_currencies`** – Defaults to a small, common-currency whitelist. Expanding it
  (e.g., adding `'JPY'`, `'CNY'`) lowers the number of flagged rows and raises the
  *Currency Validity Rate*, which may be appropriate if the platform processes global payments.
  Narrowing it increases sensitivity and flags more potential errors.

- **`iqr_multiplier`** – The standard Tukey fence uses 1.5. Reducing it to 1.0 makes outlier
  detection more aggressive (more rows flagged, higher *Outlier Rate*); increasing it to 3.0
  moves toward Extreme Outlier detection only. The right value depends on the business's
  tolerance for large transactions.

- **`amount_col` / `date_col` / `id_col`** – These string parameters generalise the function
  to any transaction schema without hard-coding column names, making it reusable across
  different database tables or file schemas with minimal configuration.


## Submission Checklist

- Dataset described  
- Issues mapped to dimensions (13 issues across Completeness, Uniqueness, Validity, Consistency, Integrity)  
- At least 3 KPIs calculated (5 KPIs computed and interpreted)  
- At least 3 validation rules defined (6 rules implemented)  
- Audit summary completed (11 issues with severity and recommended actions)  
- Cleaning recommendations proposed (6 concrete steps — not implemented)  
- Bonus: reusable `run_audit()` function with full docstring and parameter explanation  
